# CSV Files

## Introduction: What is a CSV File?

In previous chapters, you learned how enterprise databases store tabular records in structured relational tables. However, when accountants, auditors, and financial analysts extract data from corporate Enterprise Resource Planning (ERP) systems—such as SAP, Oracle, or NetSuite—or download transaction logs from banks and regulatory portals, the data is rarely delivered as a pre-formatted Excel workbook (`.xlsx`). 

Instead, bulk financial data is almost universally transmitted as a **CSV file** (`.csv`).

CSV stands for **Comma-Separated Values**. A CSV file is a plain, unformatted **text file** used to store structured tabular data. 

```
+-----------------------------------------------------------------------+
|                       PLAIN TEXT CSV FILE STRUCTURE                   |
+-----------------------------------------------------------------------+
|  Date,Stock Price,Volume                                              |
|  10/9/2019,113.86,2209500                                             |
|  10/10/2019,115.28,3083300                                            |
|  10/11/2019,116.02,1960300                                            |
+-----------------------------------------------------------------------+
```

### Why Are CSV Files So Popular?

Despite the ubiquity of Microsoft Excel, proprietary spreadsheet formats like `.xlsx` carry complex internal XML markup, binary compression, and version compatibility constraints that make them difficult to parse across different software platforms. CSV files remain the global standard for data exchange because of three distinct advantages:

1. **Platform Independence:** Because a CSV file is plain text, it can be opened and processed effortlessly on any operating system (Windows, macOS, Linux, Unix) and inside any software application (Excel, SQL Server, Python, R, Tableau, Power BI, or Canvas).
2. **Database Ingestion & Export:** Database management systems can rapidly write millions of records to a CSV file or ingest a CSV file into a database table in seconds without software overhead.
3. **Universal Data Transfer:** When moving data between two incompatible, proprietary enterprise systems (for instance, exporting payroll records from a legacy mainframe into an external audit tool), CSV serves as the universal "lingua franca" of digital accounting.

---

## How CSV Files Work: Format and Structure

Unlike an Excel workbook—which stores cell formatting, grid lines, formulas, multiple worksheets, and embedded charts—a CSV file contains strictly raw characters.

A CSV file follows a simple set of formatting rules:
* **One Line Per Row:** Every line in the text file represents exactly one row (record or observation) of tabular data.
* **Commas as Column Boundaries:** Within each line, individual column values are separated by a **comma** (`,`). The comma acts as a **delimiter**—a special character that marks where one column ends and the next column begins.
* **Text-Based Storage:** Every number, date, and text string is written out as plain text characters.

### Anatomy of a "Clean" CSV File

In financial analytics, a **"clean" CSV file** is one that satisfies four structural criteria:

1. **Header Row:** The very first line of the file contains the column names (attribute headers).
2. **Uniform Column Count:** Every single row contains the exact same number of columns. If the dataset contains $n$ columns, every line in the file must contain exactly $n - 1$ commas.
3. **Consistent Data Types:** Every value within a given column shares the same underlying data type (e.g., all dates, all numeric amounts, or all text strings). *Note:* This rule does not apply to the first header row, which always consists of text strings.
4. **Explicit Missing Values:** Missing or `NULL` entries are clearly indicated by adjacent commas without spaces between them.

### Handling Missing Values in a CSV File

Consider a stock price dataset containing dates, closing prices, and trading volumes where some days have missing volume or price entries:

| Date | Stock Price | Volume |
| :--- | :--- | :--- |
| 10/9/2019 | 113.86 | *[Missing]* |
| 10/10/2019 | 115.28 | 2,209,500 |
| 10/11/2019 | *[Missing]* | 3,083,300 |
| 10/14/2019 | 115.87 | 1,960,300 |

When saved as a CSV file, missing values are represented simply by placing two commas side by side (`,,`) or ending a line with a trailing comma (`,`):

```
Date,Stock Price,Volume
10/9/2019,113.86,
10/10/2019,115.28,2209500
10/11/2019,,3083300
10/14/2019,115.87,1960300
```

Notice that on `10/9/2019`, the trailing comma indicates that the third column (`Volume`) is missing. On `10/11/2019`, the double comma (`,,`) indicates that the second column (`Stock Price`) is missing.

---

## Delimiters & The "Comma Trap" in Accounting

While commas are the default separator in CSV files, using commas to delimit fields presents a major technical challenge when the underlying accounting data itself contains natural commas!

### The Problem: Unquoted Commas in Accounting Data

Consider a corporate vendor list containing legal business names and street addresses:
* Company Name: `Deloitte & Touche, LLP`
* Firm Name: `Dewey, Cheatem, and Howe`
* Address: `100 Main Street, Suite 400`

If an automated script exports these records into a CSV file without special handling:

```
VendorID,CompanyName,FirmName,Address
1,Deloitte & Touche, LLP,Dewey, Cheatem, and Howe,100 Main Street, Suite 400
```

When Excel or a database engine parses this line, it counts the commas to split the line into columns. Because the line contains **seven** commas instead of **three**, the parsing engine creates **eight columns** instead of **four**! 

```
+-----------------------------------------------------------------------------------+
|                        PARSING ERROR CAUSED BY UNQUOTED COMMAS                    |
+-----------------------------------------------------------------------------------+
| Column 1 (VendorID) : 1                                                           |
| Column 2 (CompanyName)  : Deloitte & Touche                                       |
| Column 3 (Shifted!) :  LLP                                                        |
| Column 4 (FirmName) : Dewey                                                       |
| Column 5 (Shifted!) :  Cheatem                                                    |
| Column 6 (Shifted!) :  and Howe                                                   |
| Column 7 (Shifted!) : 100 Main Street                                             |
| Column 8 (Shifted!) :  Suite 400                                                  |
+-----------------------------------------------------------------------------------+
```

In accounting, this type of parsing error leads to severe data corruption: vendor names are chopped in half, address details spill into credit limit columns, and financial amounts shift out of alignment across the grid.

### The Solution: Text Enclosure & Alternative Delimiters

Modern accounting systems handle commas inside text fields using two techniques:

1. **Text Enclosure (Double Quotes):** Whenever a text field contains an internal comma, the exporting software wraps the entire text string inside **double quotes** (`"..."`). When Excel encounters double quotes, it treats everything inside the quotes as a single text value, ignoring internal commas:
   ```
   VendorID,CompanyName,FirmName,Address
   1,"Deloitte & Touche, LLP","Dewey, Cheatem, and Howe","100 Main Street, Suite 400"
   ```
2. **Alternative Delimiters:** In international accounting or specialized data pipelines, files may use alternative characters to separate columns instead of commas:
   * **Tab-Separated Values (`.tsv` or `.txt`):** Uses a tab character (`\t`) between columns.
   * **Semicolon-Delimited (`.csv`):** Frequently used in European financial systems where the comma is used as a decimal separator (e.g., `1.250,50 €`).
   * **Pipe-Delimited (`.txt`):** Uses the vertical pipe symbol (`|`) to eliminate ambiguity in complex text fields.

---

## Exercise: Working with CSV Files in a Text Editor

To truly understand what a CSV file is, accountants must recognize that despite its green Excel icon on your computer desktop, **a CSV file is NOT an Excel file.**

### Exercise Instructions:

1. Download the [Excel file containing stock price data](data/UPSStockPrices.xlsx) and open it in Microsoft Excel.
2. Apply bold headings, dollar sign currency formatting (`$115.28`), highlighted cell fill colors, and a simple sum formula (`=SUM(...)`).
3. Click File $\rightarrow$ Save As and change the file type dropdown from *Excel Workbook (*.xlsx)* to **CSV (Comma delimited) (*.csv)**. Save the file as `UPS Stock Prices.csv`.
5. Close Microsoft Excel completely.
6. Open your desktop text editor (**Notepad** on Windows, **TextEdit** on Mac, or **Visual Studio Code**).
7. Inside the text editor, open `UPS Stock Prices.csv`.

### What Do You Observe in the Text Editor?

* **All visual formatting is gone:** The dollar signs, bold headings, and fill colors have vanished.
* **All formulas are gone:** If cell `B6` contained `=SUM(B2:B5)`, the text file stores only the static calculated number (e.g., `461.63`).
* **It is plain text:** You can edit numbers or delete characters directly using your keyboard!
* Try modifying a stock price on line 5 to `1.00` and deleting a missing price on line 2 (keeping the comma). Save the file in your text editor.
* Re-open `UPS Stock Prices.csv` in Microsoft Excel.

Notice that Excel opens the modified text file seamlessly, rendering your text edits in the grid cells!

---

## Opening vs. Importing CSV Files in Excel: An Auditor's Warning

In corporate accounting, double-clicking a CSV file to open it in Excel feels natural, but it carries immense operational risks.

```{warning}
**The Auditor's Perspective: Data Corruption Traps When Opening CSV Files**  
When you double-click a CSV file, Microsoft Excel automatically executes its default text-parsing algorithm to guess the data type of every column. This default behavior introduces two major audit risks:

1. **Catastrophic Truncation of Leading Zeros:**
   Excel assumes that any column containing only digits is a numeric value. If your dataset contains bank routing numbers, customer account IDs, ZIP codes, Federal Information Processing System (FIPS) codes, or Social Security Numbers with leading zeros (e.g., `00123456`), Excel automatically strips the leading zeros and converts the entry to the integer `123456`!
   
   In fraud investigations or bank reconciliations, losing leading zeros breaks record matching and corrupts account identifiers permanently if saved back to disk.

2. **Accidental Saving and Loss of Work:**
   If an analyst opens a CSV file in Excel, builds complex formulas, creates Pivot Tables, or formats balance sheet headings, and then clicks **Save** (`Ctrl + S`), Excel saves the file back in `.csv` format!
   
   Because `.csv` files store only raw text, **Excel silently discards all formulas, formatting, charts, and secondary worksheet tabs upon closing.** When the analyst reopens the file the next morning, hours of work have disappeared.

**The Power-User Rule:** Always import raw CSV files into a formal Excel Table or Power Query data model, and immediately save your working file as an **Excel Workbook (`.xlsx`)**.
```

## Summary Reference Checklist

Understanding the mechanics of CSV files ensures that you can ingest, inspect, and clean raw corporate data exports without falling into data corruption traps.

### Key Chapter Takeaways

1. **CSV Files are Plain Text:** A CSV file is a platform-independent text file where lines represent rows and commas delimit columns.
2. **CSV Stores Data Only:** CSV files cannot store formulas, cell formatting, highlights, charts, or multiple worksheet tabs.
3. **Beware of Unquoted Commas:** Commas embedded inside text fields (like vendor names or addresses) cause column-shifting errors unless enclosed in double quotes (`"..."`).
4. **Avoid the Leading Zero Trap:** Double-clicking a CSV in Excel can silently strip leading zeros from bank accounts and ID numbers. Use Excel's Data $\rightarrow$ Get Data (Power Query) import pipeline to lock ID columns as text.
5. **Always Save Working Files as `.xlsx`:** After importing a CSV into Excel, immediately save your file as an Excel Workbook (`.xlsx`) to preserve your formulas, tables, and analytical models.

### Comparison: Excel Workbook (`.xlsx`) vs. CSV Text File (`.csv`)

| Feature / Capability | Excel Workbook (`.xlsx`) | CSV Text File (`.csv`) |
| :--- | :--- | :--- |
| **File Type** | Binary / XML Container | Plain Text File |
| **Platform Compatibility** | Requires Excel or compatible spreadsheet software | Universal (Reads on any OS, software, or script) |
| **Formulas & Functions** | Supported (`=SUM()`, `=XLOOKUP()`) | **Not Supported** (Stores static text values only) |
| **Cell Formatting & Colors** | Supported (Currency, Dates, Fills, Fonts) | **Not Supported** (Unformatted text characters only) |
| **Multiple Worksheets (Tabs)**| Supported | **Not Supported** (Single table grid only) |
| **Leading Zero Preservation**| Preserved when formatted as Text | Stripped automatically if opened by double-clicking |
| **Primary Accounting Role** | Financial modeling, reporting, & dashboarding | Data extraction, ERP exchange, & database ingestion |